---
title: "01 · Establish trust in the data"
---

The unit of observation is a breast-mass sample. Before modeling, I need to establish the source, the meaning of the label, and whether the table meets a documented contract. The pipeline is a prerequisite: run `uv run cancer-study run` from a clean checkout before these notebooks. Each notebook then executes independently from a fresh kernel.

In [1]:
from cancer_study.common import root, read_json
from IPython.display import SVG, display
import pandas as pd

project = root()
metrics_dir = project / "reports/metrics"
def figure(name):
    display(SVG(filename=str(project / "site/assets" / name)))


## Preserve the source

The accepted UCI snapshot is retained with a source manifest. A changed checksum stops processing rather than silently accepting a different dataset.

In [2]:
from cancer_study.ingest import ingest
from cancer_study.datasets import assign_splits, model_inputs

frame, quality = ingest(project)
partitioned = assign_splits(project, frame)
source = read_json(project / "data/source.json")
display(pd.Series(source, name="source manifest"))

url                 https://archive.ics.uci.edu/static/public/17/b...
retrieved_at_utc                     2026-10-26T19:29:57.455585+00:00
sha256              d606af411f3e5be8a317a5a8b652b425aaf0ff38ca683d...
license                                                     CC BY 4.0
citation            Wolberg, W., Mangasarian, O., Street, N., & St...
file                                               data/raw/wdbc.data
observation_unit                               one breast-mass sample
Name: source manifest, dtype: object

## Validate before transforming

No values are imputed and no rows are removed. Zeros are allowed. Deliberately malformed test fixtures demonstrate rejection behavior; they are not presented as defects in the original data.

In [3]:
display(pd.Series({k: v for k, v in quality.items() if k != "checks"}, name="quality report"))
display(pd.DataFrame({"check": quality["checks"], "status": "passed"}))

status                       passed
rows                            569
measurement_columns              30
missing_values                    0
duplicate_identifiers             0
duplicate_feature_vectors         0
nonfinite_values                  0
negative_measurements             0
benign                          357
malignant                       212
zero_measurements                78
Name: quality report, dtype: object

,check,status
0,column count and order,passed
1,nonempty source,passed
2,complete values,passed
3,valid labels,passed
4,both classes present,passed
5,unique numeric identifiers,passed
6,numeric finite measurements,passed
7,nonnegative measurements,passed
8,no duplicate feature vectors,passed


## Persist the partition

The stratified 80/20 split is assigned once with seed 42 and saved by sample ID. Source validation covers the ingestion; exploration that informs choices uses development data. The source has no duplicate vectors, though the available metadata cannot establish patient independence beyond sample IDs.

In [4]:
display(partitioned.groupby(["partition", "diagnosis"]).size().unstack(fill_value=0))
x, y = model_inputs(partitioned)
assert x.shape[1] == 30
assert not {"sample_id", "diagnosis", "target", "partition"} & set(x.columns)
assert y.eq(1).equals(partitioned.diagnosis.eq("M"))
print("Identifier and label excluded. Malignant is positive.")

diagnosis,B,M
partition,,
development,285,170
holdout,72,42


Identifier and label excluded. Malignant is positive.


## Query the validated table

DuckDB provides a readable SQL layer over the same validated rows. Cohort counts include both partitions for accounting; feature summaries used in exploration are restricted to development.

In [5]:
import duckdb

with duckdb.connect() as connection:
    connection.register("samples", partitioned)
    sql = (project / "sql/cohort_summary.sql").read_text()
    print(sql)
    display(connection.execute(sql).df())

SELECT partition,
       CASE WHEN target = 1 THEN 'Malignant' ELSE 'Benign' END AS diagnosis,
       count(*) AS samples,
       round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY partition), 2) AS percent_within_partition
FROM samples
GROUP BY partition, target
ORDER BY partition, target;



,partition,diagnosis,samples,percent_within_partition
0,development,Benign,285,62.64
1,development,Malignant,170,37.36
2,holdout,Benign,72,63.16
3,holdout,Malignant,42,36.84


## Decision

The accepted source passes the contract, so retain its observations and measurements as supplied. Exclude the identifier from predictors, keep malignant as the positive class, and preserve the saved split for all later work.